# Why does the 8km JRA RYF run drift? (8km test2 vs 25km RYF)

**Status:** the spurious deep mixing is identified. A one-change test with a shorter tracer/thermodynamic time step removes most of it and about half of the extra heat uptake, but not all of the global drift (Part E). Each statement below points to the section that shows it. The main comparison is with **25km test5.1**, which is built from the **same executable** as 8km.

## What goes wrong
- **The 8km run warms 10-20 times faster than 25km, from the first year (A1, A2).** Global-mean T rises by +0.019 to +0.030 °C per year at 8km, against +0.001 °C (25km test5.1) and +0.002 °C (25km test3). From 1903 the 8km rate holds steady at about +0.020 °C per year.
- **It diverges within weeks (A3, A4).** The global-mean T gap from 25km is +0.0002 °C at day 30, +0.0009 °C at day 60 and +0.019 °C at day 365. The extra net surface heat uptake at 8km grows from +0.5 PW in January 1900 to about +5 PW in December.

## Where and how (Part B, all in 1900)
1. **It starts in the western-boundary-current extensions (B1).** By month 3, the 8km run has cooled by 1-3 °C at 300 m in the Gulf Stream, Kuroshio, Agulhas and EAC extensions. The 25km run barely changes there, and the gyre interiors stay the same in both runs.
2. **Heat moves from the thermocline to the abyss (B2).** In all four current-extension boxes, 8km loses 3-7 GJ m⁻² from 100-700 m and gains 2-6 GJ m⁻² below 1500 m between January and June. That is about 0.3 °C of deep warming in five months. The 25km deep change is about zero.
3. **It is not convection, and not the parameterised mixing (B3).** Mixed layers are similar in both runs and no deeper than about 500 m. Parameterised `Kd` below 700 m is almost the same in both runs, dominated by the internal-tide term.
4. **Water changes density class in the 8km interior (B4).** The global volume of the densest water (σ₂ > 37.0) shrinks by 4.6% in five months (528.7 → 504.4 × 10¹⁵ m³), at an increasing rate, while intermediate classes grow. 25km stays flat. The loss is in the current extensions and the ACC band (40-60°S), not at the Antarctic margin where these densities outcrop. In MOM6, the resolved flow cannot change a water mass's density class, and surface fluxes can only do so where that density reaches the surface. With parameterised mixing unchanged, the remaining explanation is **numerical (spurious) diapycnal mixing at 8km in energetic current regions**.
5. **The advection scheme's numerical mixing is much larger at 8km in these regions (B5).** It is 10²-10⁵ times the 25km value, mostly below 700 m. Remapping (`T_vardec`) is small in both runs. *Caveat:* 8km has no explicit lateral tracer diffusion (`KHTR` = 0, no GM), so part of this difference is expected, and these diagnostics do not by themselves split diapycnal from lateral mixing.
6. **The 8km tracer time step runs closer to the CFL limit (B6).** From month 3, the 99th-percentile surface CFL in the current extensions is about 0.7-0.8 at 8km against about 0.35-0.4 at 25km.

**Reading:** the cooled thermocline in the current extensions, and later the equatorial changes, draw extra heat in at the surface, and that drives the global drift. The open question is *why* the 8km numerics mix so much there. Part E shows that the tracer/thermodynamic time step is a strong lever. Other possible contributors: grid-scale noise (`KHTR` = 0, no GM, viscosity settings), or something else.

## Ruled out (with the evidence)
| Candidate | Evidence | Section |
|---|---|---|
| Executable / build | 25km test5.1 uses the same `pr264-25` executable as 8km and does not drift | A1, A2 |
| Wind remapping | 1909 stress pattern correlation about 0.99 on a 1° grid; at native resolution, rms curl is similar and 8km grid-scale curl noise is lower | C1, C2 |
| Initial conditions | Jan 1900 zonal-mean T agrees with 25km to within a few hundredths of a °C; no 300 m differences in month 1 | C3, B1 |
| Vertical grid | Same z* coordinate; the two `ocean_vgrid.nc` files have identical interfaces | C4 |
| Background diffusivity `KD` | 8km 1.5e-5 vs test5.1 8e-6, but 25km test3 also uses 1.5e-5 and does not drift | C4 |
| `NONBLOCKING_P_SURF_BUG` | Active in 8km 1900-1904 and switched off from 1905, with no change in warming rate | A2, C4 |
| Dynamics time-step ramp | `DT` raised in 1902 and 1904 with no step in warming rate; divergence already present at `DT` = 240 s | A2, A3, C4 |
| Mid-run config/exe changes | Exe and input manifests identical in all 32 segments | C4 |
| Convection / parameterised mixing | Similar MLD and `Kd` below 700 m in the current-extension boxes | B3 |

## One-change test: `DT_THERM` 3600 → 2400 s (Part E)
This repeats test2 exactly from the same WOA23 start, changing only `DT_THERM`; `DT_TRACER_ADVECT` follows it by default. Each segment copies test2's: months 1-2 at `DT` = 240 s, months 3-4 at `DT` = 400 s. The months 5-6 segment failed at day 166 on a transient NetCDF read error in an input file (the file was checked afterwards and reads cleanly), so no 3D output exists after month 4. Its `ocean.stats` survived and is used in point 2 below.

**1. The spurious deep mixing in the current extensions is almost gone (E2).**

| Month 4 | 25km test5.1 | 8km `DT_THERM` 2400 | 8km test2 |
|---|---|---|---|
| Change in global V(σ₂ > 37.0) since month 1 (10¹⁵ m³) | +0.04 | **−0.18** | −10.6 |
| Gulf Stream advection numerical mixing >700 m (K² m s⁻¹) | −1.6e-4 | **−3.2e-3** | −2.2e-1 |
| Gulf Stream box T at 300 m (°C) | 15.99 | **15.91** | 13.38 |
| Agulhas box T at 300 m (°C) | 11.54 | **11.46** | 9.84 |

A one-third shorter tracer step cuts the numerical mixing about 70-fold, which suggests a threshold-like (CFL-related) behaviour rather than a gradual one.

**2. The global drift is reduced, not removed (E1, E3).**
- The global-mean T gap to 25km is about 40% smaller at day 118 (+0.0015 °C against +0.0026 °C for test2) and about 57% smaller at day 166 (+0.0021 °C against +0.0049 °C), from the failed segment's `ocean.stats`. Over days 120-166 the gap grew by +0.0006 °C in the test and +0.0023 °C in test2.
- The excess surface heat uptake in month 4 is about 60% smaller: +0.87 PW against +2.10 PW.
- What goes is mostly the mid-latitude excess (45-20°S: 0.77 → 0.22 PW; 20-45°N: 0.54 → 0.14 PW), at the latitudes of the current extensions.
- What stays:
  - the tropical excess (20°S-20°N, about +0.2 PW in both 8km runs);
  - about half the Southern Ocean excess;
  - an excess of about 0.4 PW already present in month 1 and the same in both 8km runs.
- Global KE is still well above 25km. At day 166 it is 2.25e-3 in the test, levelling off, against 4.13e-3 in test2, still rising, and 1.30e-3 at 25km.

**Not yet known:**
- Whether the remaining excess is a second problem, or just what a different resolution should be expected to do.
- Which of `DT_THERM` or `DT_TRACER_ADVECT` matters.
- Whether an even shorter step removes more.

## Movie
Part D shows the EAC region in both runs for 1900-1907.


In [1]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
# (the 8km run has no datastore yet, so files are read directly below; esm_file is only used for naming)
esm_file = "/g/data/ol01/outputs/access-om3-8km/MC_8km_jra_ryf-test2-expt-c349f800/experiment_datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name

In [2]:
# Parameters
papermill = True
cwd = "/g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/"
nbname = "8km_ryf_drift_diagnosis.ipynb"


In [3]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

## Setup
Runs compared:
- **8km test2**: the drifting run.
- **25km test5.1**: built from the **same executable** as 8km (`access-om3/pr264-25`, hash `mfvxyjb`); 1900-1907.
- **25km test3**: the original 25km RYF control (older build); 1900-1930. Used for the 1909 comparisons, because test5.1 stops at 1907.

All three start from WOA23 at rest on 1900-01-01 and are forced by the same JRA55-do v1.6 RYF9091 streams.

In [4]:
import glob, os, re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

E8 = os.path.dirname(esm_file)
E25 = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_ryf+wombatlite-test3-f4d79e82"
E25S = "/g/data/ik11/outputs/access-om3-025/MC_25km_jra_ryf+wombatlite-test5.1-5c6ad028"   # same exe as 8km
RUNS = {"8km": E8, "25km": E25}                       # 1909 comparisons
ALL = {"8km test2": E8, "25km test5.1 (same exe)": E25S, "25km test3": E25}
YEAR = 1909
YB = np.arange(-80, 91, 1.0)   # 1-degree latitude bins for zonal means

def files(exp, pat, year=YEAR, freq="1mon.mean"):
    return sorted(glob.glob(f"{exp}/output*/access-om3.mom6.{pat}.{freq}.{year}.nc"))

def annual(exp, var, sel=None, dim="2d"):
    # Annual mean of a monthly diagnostic for YEAR (files are split across output dirs).
    name = var.split("+")[0]
    das = []
    for f in files(exp, f"{dim}.{var}"):
        da = xr.open_dataset(f, decode_timedelta=False)[name]
        if sel:
            da = da.sel(sel, method="nearest")
        das.append(da.load())
    da = xr.concat(das, "time").drop_duplicates("time")
    assert da.sizes["time"] == 12, (exp, var, da.sizes["time"])
    return da.mean("time")

def static(exp):
    return xr.open_dataset(sorted(glob.glob(f"{exp}/output*/access-om3.mom6.static.nc"))[0])

def bin_lat_lon(da, res=1.0):
    # Bin a 2D field onto a res-degree grid using nominal 1D coords (fine south of 65N).
    x = [d for d in da.dims if d.startswith("x")][0]; y = [d for d in da.dims if d.startswith("y")][0]
    LON, LAT = np.meshgrid((da[x].values + 360) % 360, da[y].values)
    v = da.values; m = np.isfinite(v)
    xb = np.arange(0, 360 + res, res); yb = np.arange(-80, 90 + res, res)
    s, _, _ = np.histogram2d(LAT[m], LON[m], [yb, xb], weights=v[m])
    n, _, _ = np.histogram2d(LAT[m], LON[m], [yb, xb])
    return xr.DataArray(np.where(n > 0, s / np.maximum(n, 1), np.nan), dims=("lat", "lon"),
                        coords=dict(lat=yb[:-1] + res / 2, lon=xb[:-1] + res / 2))

def area_lat(exp):
    # Wet-cell area and nominal latitude on tracer points.
    st = static(exp)
    A = (st.areacello.fillna(0) * st.wet.fillna(0)).values
    return A, np.broadcast_to(st.yh.values[:, None], A.shape)

def zm(t, A, lat):
    # Area-weighted zonal mean of one 2D field in 1-degree nominal-latitude bins.
    m = np.isfinite(t) & (np.abs(t) < 1e10) & (A > 0)
    s, _ = np.histogram(lat[m], YB, weights=(t * A)[m]); n, _ = np.histogram(lat[m], YB, weights=A[m])
    return np.where(n > 0, s / np.maximum(n, 1e-9), np.nan)

def zm3(da, A, lat):
    # Zonal mean of a (z, y, x) field, one level at a time to keep memory small.
    zd = da.dims[0]
    out = np.array([zm(da.isel({zd: k}).values.astype(float), A, lat) for k in range(da.sizes[zd])])
    return xr.DataArray(out, dims=("z", "lat"), coords=dict(z=da[zd].values, lat=YB[:-1] + 0.5))

def month_index(pattern):
    # (file, time index) for each unique month across output segments, in time order (no concatenation).
    seen, out = set(), []
    for f in sorted(glob.glob(pattern)):
        with xr.open_dataset(f, decode_timedelta=False) as ds:
            for i, t in enumerate(ds.time.values):
                if t not in seen:
                    seen.add(t); out.append((f, i, t))
    return sorted(out, key=lambda x: x[2])

def stats(exp):
    # ocean.stats: day, global-mean T, KE per unit mass (Energy/Mass, m2 s-2), duplicates removed.
    rows = []
    for f in sorted(glob.glob(f"{exp}/output*/ocean.stats")):
        for line in open(f).readlines()[2:]:
            p = line.replace(",", " ").split()
            rows.append((float(p[1]), float(p[p.index("T") + 1]), float(p[p.index("En") + 1])))
    a = np.unique(np.array(rows), axis=0)
    return a[:, 0] - a[0, 0], a[:, 1], a[:, 2]

# Part A: what goes wrong

### A1. Global-mean temperature and kinetic energy (`ocean.stats`)

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ST = {k: stats(E) for k, E in ALL.items()}
for k, (day, T, En) in ST.items():
    ax[0].plot(1900 + day / 365, T, label=k); ax[1].plot(1900 + day / 365, En, label=k)
ax[0].set(title="global-mean T", ylabel="°C", xlim=(1900, 1911))
ax[1].set(title="global KE per unit mass (ocean.stats Energy/Mass)", ylabel="m$^2$ s$^{-2}$", xlim=(1900, 1911))
for a in ax:
    a.axvline(1905, color="0.5", ls=":", lw=1)
    a.legend()
ax[0].text(1905.1, ax[0].get_ylim()[0], " 8km: NONBLOCKING_P_SURF_BUG\n True -> False", fontsize=8, va="bottom")
plt.tight_layout()
mkmd.savefig(fig, "8km global drift", "Global-mean T and KE from ocean.stats. The 8km run warms about 10-20 times faster than both 25km runs, including 25km test5.1 built from the same executable.", dpi=100)

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_01.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.


### A2. Warming rate by year
Change in global-mean T over each model year. The dotted line in A1, and the 1905 row here, mark the 8km segment (`output020`) where `NONBLOCKING_P_SURF_BUG` was switched from True to False (see C4). The 8km dynamics time step was also raised at the start of 1902 (400 → 600 s) and 1904 (600 → 720 s).

In [6]:
rows = {}
for k, (day, T, En) in ST.items():
    yr = 1900 + np.floor(day / 365).astype(int)
    rows[k] = {y: T[yr == y][-1] - T[yr == y][0] for y in np.unique(yr) if (yr == y).sum() > 1 and y <= 1909}
rate = pd.DataFrame(rows).round(4)
rate.index.name = "year"
rate.style.format("{:+.4f}", na_rep="")

,8km test2,25km test5.1 (same exe),25km test3
year,,,
1900,+0.0190,+0.0000,+0.0015
1901,+0.0298,+0.0003,+0.0015
1902,+0.0273,+0.0008,+0.0019
1903,+0.0235,+0.0009,+0.0019
1904,+0.0210,+0.0007,+0.0021
1905,+0.0201,+0.0008,+0.0020
1906,+0.0205,+0.0010,+0.0020
1907,+0.0198,+0.0010,+0.0021
1908,+0.0195,+0.0009,+0.0021


### A3. When does the 8km run start to diverge?
Global-mean T change since 1 Jan 1900, 8km test2 vs 25km test5.1 (same executable). The gap is already visible after about a month and grows steadily.

In [7]:
days = [5, 10, 15, 20, 30, 45, 60, 90, 120, 150, 180, 240, 300, 365]
d8, T8, _ = ST["8km test2"]; d25, T25, _ = ST["25km test5.1 (same exe)"]
early = pd.DataFrame({"8km dT (°C)": np.interp(days, d8, T8 - T8[0]),
                      "25km test5.1 dT (°C)": np.interp(days, d25, T25 - T25[0])}, index=pd.Index(days, name="day"))
early["8km - 25km (°C)"] = early.iloc[:, 0] - early.iloc[:, 1]
early.style.format("{:+.5f}")

,8km dT (°C),25km test5.1 dT (°C),8km - 25km (°C)
day,,,
5,+0.00060,+0.00060,+0.00000
10,+0.00120,+0.00110,+0.00010
15,+0.00170,+0.00160,+0.00010
20,+0.00220,+0.00210,+0.00010
30,+0.00330,+0.00310,+0.00020
45,+0.00480,+0.00430,+0.00050
60,+0.00565,+0.00480,+0.00085
90,+0.00790,+0.00620,+0.00170
120,+0.00850,+0.00590,+0.00260


### A4. 1900 month by month: surface heat flux and zonal-mean T (8km test2 vs 25km test5.1)
`hfds` is the net surface heat flux into the ocean. T sections are read one month and one level at a time, so this cell is slow (about 15-20 min) but needs little memory.

In [8]:
MONTHS = [0, 1, 2, 5, 11]
M = {}
for k, E in {"8km": E8, "25km": E25S}.items():
    A, lat = area_lat(E)
    hz, hg = [], []
    for f, i, _ in month_index(f"{E}/output*/access-om3.mom6.2d.hfds.1mon.mean.1900.nc"):
        with xr.open_dataset(f, decode_timedelta=False) as ds:
            h = ds.hfds.isel(time=i).values
        hz.append(zm(h, A, lat)); hg.append(np.nansum(h * A) / 1e15)
    M[(k, "hfds")], M[(k, "hfds_glob")] = np.array(hz), np.array(hg)
    tidx = month_index(f"{E}/output*/access-om3.mom6.3d.thetao+z.1mon.mean.1900.nc")
    for mo in MONTHS:
        f, i, _ = tidx[mo]
        with xr.open_dataset(f, decode_timedelta=False) as ds:
            M[(k, "T", mo)] = zm3(ds.thetao.isel(time=i).sel(z_l=slice(0, 1500)), A, lat)

pd.DataFrame({k: M[(k, "hfds_glob")] for k in ["8km", "25km"]}, index=pd.Index(range(1, 13), name="month 1900")).round(2).rename(
    columns=lambda c: f"{c} global hfds (PW)")

,8km global hfds (PW),25km global hfds (PW)
month 1900,,
1,6.97,6.50
2,4.89,3.70
3,4.71,2.98
4,1.38,-0.72
5,-3.29,-6.22
6,-5.33,-8.83
7,-2.60,-6.56
8,1.79,-2.78
9,5.59,1.07


In [9]:
lat_c = YB[:-1] + 0.5
fig, ax = plt.subplots(2, 3, figsize=(16, 8)); ax = ax.ravel()
d = M[("8km", "hfds")] - M[("25km", "hfds")]
c = ax[0].pcolormesh(lat_c, np.arange(1, 13), d, vmin=-60, vmax=60, cmap="RdBu_r"); fig.colorbar(c, ax=ax[0], label="W m$^{-2}$")
ax[0].set(title="zonal-mean hfds 8km - 25km test5.1, 1900", xlabel="lat", ylabel="month", xlim=(-80, 70))
for i, mo in enumerate(MONTHS):
    dT = M[("8km", "T", mo)] - M[("25km", "T", mo)].interp(z=M[("8km", "T", mo)].z)
    dT.plot.contourf(ax=ax[i + 1], levels=np.arange(-2, 2.01, 0.2), cmap="RdBu_r", extend="both", yincrease=False,
                     cbar_kwargs=dict(label="°C"))
    ax[i + 1].set(title=f"zonal-mean T 8km - 25km test5.1, month {mo + 1} of 1900", xlim=(-80, 70), ylabel="depth (m)")
plt.tight_layout()
mkmd.savefig(fig, "1900 monthly divergence", "Zonal-mean surface heat flux (8km minus 25km test5.1, same executable) for each month of 1900, and zonal-mean T differences after 1, 2, 3, 6 and 12 months.", dpi=100)
eq = (lat_c > -10) & (lat_c < 10)
print("8km - 25km hfds, 10S-10N mean by month (W/m2):", np.nanmean(d[:, eq], 1).round(1))

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_02.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.
8km - 25km hfds, 10S-10N mean by month (W/m2): [ 0.9  1.6  1.7  2.1  5.4 11.  15.  19.9 20.1 19.3 21.7 22.2]


### A5. Zonal-mean T difference after ten years (8km - 25km test3, Dec 1909)

In [10]:
def zonal_mean_T(exp):
    f = files(exp, "3d.thetao+z")[-1]
    A, lat = area_lat(exp)
    return zm3(xr.open_dataset(f, decode_timedelta=False).thetao.isel(time=-1), A, lat)

zmT = {k: zonal_mean_T(E) for k, E in RUNS.items()}
fig, ax = plt.subplots(figsize=(10, 4.5))
(zmT["8km"] - zmT["25km"]).plot.contourf(ax=ax, levels=np.arange(-3, 3.1, 0.25), cmap="RdBu_r", yincrease=False,
                                         cbar_kwargs=dict(label="°C"))
zmT["25km"].plot.contour(ax=ax, levels=range(0, 30, 2), colors="k", linewidths=0.5, yincrease=False)
ax.set(title="zonal-mean T, 8km - 25km test3 (Dec 1909); contours: 25km T", ylabel="depth (m)")
plt.tight_layout()
mkmd.savefig(fig, "8km thermocline cold / deep warm", "Zonal-mean temperature 8km minus 25km, Dec 1909. The 8km thermocline is colder and the deeper water warmer.", dpi=100)

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_03.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.


### A6. Agulhas: surface speed and SSH (1909 annual mean)

In [11]:
def agulhas(da):
    x = [d for d in da.dims if d.startswith("x")][0]; y = [d for d in da.dims if d.startswith("y")][0]
    da = da.assign_coords({x: ((da[x] + 180) % 360) - 180}).sortby(x)
    return da.sel({x: slice(0, 60), y: slice(-50, -20)})

fig, ax = plt.subplots(2, 2, figsize=(13, 7))
for c, (k, E) in enumerate(RUNS.items()):
    agulhas(annual(E, "speed")).plot(ax=ax[0, c], vmin=0, vmax=1.2, cmap="magma"); ax[0, c].set_title(f"{k} surface speed")
    z = agulhas(annual(E, "zos"))
    z.plot(ax=ax[1, c], vmin=-1, vmax=1, cmap="RdBu_r")
    z.plot.contour(ax=ax[1, c], levels=np.arange(-1, 1.01, 0.1), colors="k", linewidths=0.4); ax[1, c].set_title(f"{k} SSH")
plt.tight_layout()

# Part B: where and how the 8km run moves heat (1900, 8km test2 vs 25km test5.1, same executable)

### B1. Temperature at 300 m
The first large differences are confined to the western-boundary-current extensions. The gyre interiors stay the same.

In [12]:
Z300 = 300.0; XB2 = np.arange(0, 361, 1.0); YB2 = np.arange(-80, 71, 1.0)
BOX = {"Gulf Stream": (280, 320, 30, 45), "Kuroshio": (135, 175, 28, 42), "Agulhas": (10, 60, -45, -33),
       "EAC": (150, 170, -45, -28), "Eq Pacific": (150, 270, -5, 5)}
WBC = ["Gulf Stream", "Kuroshio", "Agulhas", "EAC"]
P2 = {"8km": E8, "25km": E25S}     # Part B pairs 8km with the same-executable 25km run

def bin2(da, xb=XB2, yb=YB2):
    LON, LAT = np.meshgrid((da.xh.values + 360) % 360, da.yh.values); v = da.values.astype(float); m = np.isfinite(v)
    s, _, _ = np.histogram2d(LAT[m], LON[m], [yb, xb], weights=v[m]); n, _, _ = np.histogram2d(LAT[m], LON[m], [yb, xb])
    return np.where(n > 0, s / np.maximum(n, 1), np.nan)

def box_slices(E, x0, x1, y0, y1):
    st = static(E); lon = (st.xh.values + 360) % 360; lat = st.yh.values
    ix = np.where((lon >= x0) & (lon < x1))[0]; iy = np.where((lat >= y0) & (lat < y1))[0]
    return slice(ix.min(), ix.max() + 1), slice(iy.min(), iy.max() + 1)

T300 = {}
for k, E in P2.items():
    idx = month_index(f"{E}/output*/access-om3.mom6.3d.thetao+z.1mon.mean.1900.nc")
    for mo in [0, 2, 5]:
        f, i, _ = idx[mo]
        with xr.open_dataset(f, decode_timedelta=False) as ds:
            T300[(k, mo)] = bin2(ds.thetao.isel(time=i).sel(z_l=Z300, method="nearest"))
fig, ax = plt.subplots(3, 1, figsize=(13, 13))
for j, mo in enumerate([0, 2, 5]):
    c = ax[j].pcolormesh(XB2, YB2, T300[("8km", mo)] - T300[("25km", mo)], vmin=-4, vmax=4, cmap="RdBu_r")
    fig.colorbar(c, ax=ax[j], label="°C"); ax[j].set(title=f"T at 300 m, 8km - 25km test5.1, month {mo + 1} of 1900", ylabel="lat")
plt.tight_layout()
mkmd.savefig(fig, "300 m T difference 1900", "Temperature at 300 m, 8km minus 25km test5.1 (same executable), after 1, 3 and 6 months. The differences start in the western-boundary-current extensions.", dpi=90)

rows = []
for nm in BOX:
    x0, x1, y0, y1 = BOX[nm]; jy = (YB2[:-1] >= y0) & (YB2[:-1] < y1); ix = (XB2[:-1] >= x0) & (XB2[:-1] < x1)
    b = lambda a: np.nanmean(a[np.ix_(jy, ix)])
    rows.append({"box": nm, **{f"{k} change m1->m{mo + 1} (°C)": b(T300[(k, mo)]) - b(T300[(k, 0)]) for k in P2 for mo in [2, 5]}})
print("Box-mean change at 300 m since month 1, each run separately:")
pd.DataFrame(rows).set_index("box").round(2)

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_04.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.
Box-mean change at 300 m since month 1, each run separately:


,8km change m1->m3 (°C),8km change m1->m6 (°C),25km change m1->m3 (°C),25km change m1->m6 (°C)
box,,,,
Gulf Stream,-1.28,-3.29,0.18,0.30
Kuroshio,-0.48,-2.00,0.05,0.24
Agulhas,-0.83,-2.70,0.06,0.12
EAC,-0.27,-1.25,0.07,0.11
Eq Pacific,0.05,0.60,-0.03,-0.09


### B2. Heat budget of the current-extension boxes, by depth (month 1 → month 6 of 1900)
Heat content comes from monthly-mean `thetao` on z levels (ρc_p = 1035 × 3992 J m⁻³ K⁻¹; partial bottom cells ignored). Surface input is the monthly `hfds` integrated between month mid-points. The lateral term is the residual: heat-content change minus surface input.

In the 8km run, all four current-extension boxes move heat from 100-700 m to below 1500 m. The 25km run's deep change is about zero.

In [13]:
RHO_CP = 1035.0 * 3992.0
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]) * 86400.0
BANDS = [(0, 100), (100, 300), (300, 700), (700, 1500), (1500, 7000)]
rows, brows = [], []
for k, E in P2.items():
    A, _ = area_lat(E)
    tidx = month_index(f"{E}/output*/access-om3.mom6.3d.thetao+z.1mon.mean.1900.nc")
    hidx = month_index(f"{E}/output*/access-om3.mom6.2d.hfds.1mon.mean.1900.nc")
    for nm in BOX:
        sx, sy = box_slices(E, *BOX[nm]); a = A[sy, sx]; area = a.sum(); prof = {}
        for mo in [0, 5]:
            f, i, _ = tidx[mo]
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                T = ds.thetao.isel(time=i, yh=sy, xh=sx).values.astype(float); dz = np.diff(ds.z_i.values); z = ds.z_l.values
            prof[mo] = RHO_CP * np.nansum(T * a[None], axis=(1, 2)) * dz / area / 1e9        # GJ m-2 per level
        F = []
        for f, i, _ in hidx[:6]:
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                F.append(np.nansum(ds.hfds.isel(time=i, yh=sy, xh=sx).values * a))
        surf = sum(0.5 * (F[j] + F[j + 1]) * 0.5 * (DAYS[j] + DAYS[j + 1]) for j in range(5)) / area / 1e9
        d = prof[5] - prof[0]
        rows.append({"box": nm, "run": k, "dH total": d.sum(), "surface in": surf, "lateral (residual)": d.sum() - surf})
        brows.append({"box": nm, "run": k, **{f"{z0}-{z1} m": d[(z >= z0) & (z < z1)].sum() for z0, z1 in BANDS}})
print("Column heat budget, GJ m-2, month 1 -> month 6 of 1900")
display(pd.DataFrame(rows).set_index(["box", "run"]).round(2))
print("Heat-content change by depth band, GJ m-2, month 1 -> month 6 of 1900")
pd.DataFrame(brows).set_index(["box", "run"]).round(2)

Column heat budget, GJ m-2, month 1 -> month 6 of 1900


,,dH total,surface in,lateral (residual)
box,run,,,
Gulf Stream,8km,-0.82,0.08,-0.89
Kuroshio,8km,-1.13,-0.06,-1.07
Agulhas,8km,0.36,0.07,0.29
EAC,8km,-0.09,-0.52,0.43
Eq Pacific,8km,-0.03,0.88,-0.91
Gulf Stream,25km,0.72,-0.36,1.08
Kuroshio,25km,0.48,-0.32,0.79
Agulhas,25km,0.33,-0.63,0.95
EAC,25km,-0.40,-0.79,0.39


Heat-content change by depth band, GJ m-2, month 1 -> month 6 of 1900


,,0-100 m,100-300 m,300-700 m,700-1500 m,1500-7000 m
box,run,,,,,
Gulf Stream,8km,-0.16,-2.40,-4.63,1.32,5.05
Kuroshio,8km,-0.29,-2.12,-2.03,0.87,2.43
Agulhas,8km,-1.51,-2.22,-3.46,1.23,6.32
EAC,8km,-1.18,-1.06,-1.50,0.28,3.37
Eq Pacific,8km,-0.30,-0.63,0.54,0.27,0.08
Gulf Stream,25km,0.66,0.16,0.09,-0.17,-0.01
Kuroshio,25km,0.52,-0.19,0.16,-0.00,-0.01
Agulhas,25km,-0.48,0.23,0.19,0.27,0.12
EAC,25km,-0.66,0.18,0.07,0.00,0.01


### B3. Not convection, and not the parameterised mixing
In each box: the mixed-layer depth (`mlotst` monthly maximum), and the box-mean parameterised diffusivity by depth band (`Kd_interface` and its components, on native interfaces, depth from `e`). Below 700 m the two runs are almost the same, and the internal-tide term dominates.

In [14]:
KBANDS = [(100, 300), (300, 700), (700, 1500), (1500, 3000)]
rows = []
for nm in WBC:
    for k, E in P2.items():
        sx, sy = box_slices(E, *BOX[nm])
        for mo in [0, 2, 5]:
            f, i, _ = month_index(f"{E}/output*/access-om3.mom6.2d.mlotst.1mon.max.1900.nc")[mo]
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                ml = ds.mlotst_max.isel(time=i, yh=sy, xh=sx).values
            f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.e.1mon.mean.1900.nc")[mo]
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                dep = -ds.e.isel(time=i, yh=sy, xh=sx).values
            row = {"box": nm, "run": k, "month": mo + 1, "MLD max p99.9 (m)": np.nanpercentile(ml, 99.9)}
            for v in ["Kd_interface", "Kd_itides", "Kd_bkgnd"]:
                f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.{v}.1mon.mean.1900.nc")[mo]
                with xr.open_dataset(f, decode_timedelta=False) as ds:
                    kd = ds[v].isel(time=i, yh=sy, xh=sx).values
                for a0, b0 in KBANDS[2:]:
                    row[f"{v} {a0}-{b0} m"] = np.nanmean(np.where((dep >= a0) & (dep < b0), kd, np.nan))
            rows.append(row)
pd.DataFrame(rows).set_index(["box", "run", "month"]).style.format("{:.1e}").format("{:.0f}", subset=["MLD max p99.9 (m)"])

### B4. Global volume by density class: interior diapycnal change in the 8km run only
Globally, the volume denser than a given σ₂ can change only by diapycnal transport (parameterised mixing, numerical mixing, or surface buoyancy fluxes where that density outcrops). Lateral exchange cancels out. Computed from `thkcello` on the σ₂ diagnostic grid, which uses the same definition in both runs; total volume is conserved to 1e-5.

In the 8km run, the densest water (σ₂ > 37.0) gets steadily lighter while intermediate classes grow, which is the signature of diapycnal mixing. The map shows that the loss happens in the boundary-current extensions and the ACC band (40-60°S), not at the Antarctic margin where these densities outcrop.

In [15]:
SIG = [1035.5, 1036.0, 1036.8, 1037.0]
CEN = {}; H37 = {}
for k, E in P2.items():
    A, _ = area_lat(E); A = static(E).areacello.fillna(0).values
    idx = month_index(f"{E}/output*/access-om3.mom6.3d.thkcello+rho2.1mon.mean.1900.nc")
    V = []
    for mo in range(6):
        f, i, _ = idx[mo]
        with xr.open_dataset(f, decode_timedelta=False) as ds:
            h = ds.thkcello.isel(time=i); rd = h.dims[0]; rho = ds[rd].values
            lay = [np.nan_to_num(h.isel({rd: kk}).values) for kk in range(h.shape[0])]
        V.append(np.array([np.sum(l * A) for l in lay]))
        if mo in (0, 5):
            H37[(k, mo)] = sum(lay[kk] for kk in np.where(rho > 1037.0)[0]) * A
    V = np.array(V)
    CEN[k] = {f"V(σ2>{s - 1000:.1f})": V[:, rho > s].sum(1) / 1e15 for s in SIG}
    CEN[k]["total"] = V.sum(1) / 1e15
cen = pd.concat({k: pd.DataFrame(v, index=pd.Index(range(1, 7), name="month 1900")) for k, v in CEN.items()}, axis=1)
print("Global volume denser than σ2, 1e15 m3")
display(cen.round(2))

fig, ax = plt.subplots(2, 1, figsize=(13, 9)); rows = []
for j, (k, E) in enumerate(P2.items()):
    st = static(E); LON, LAT = np.meshgrid((st.xh.values + 360) % 360, st.yh.values)
    dV = H37[(k, 5)] - H37[(k, 0)]
    xb, yb = np.arange(0, 361, 2.0), np.arange(-80, 71, 2.0)
    s, _, _ = np.histogram2d(LAT.ravel(), LON.ravel(), [yb, xb], weights=dV.ravel())
    a, _, _ = np.histogram2d(LAT.ravel(), LON.ravel(), [yb, xb], weights=st.areacello.fillna(0).values.ravel())
    c = ax[j].pcolormesh(xb, yb, np.where(a > 0, s / np.maximum(a, 1), np.nan), vmin=-300, vmax=300, cmap="RdBu_r")
    fig.colorbar(c, ax=ax[j], label="m"); ax[j].set(title=f"{k}: change in thickness of water with σ2 > 37.0, Jun - Jan 1900", ylabel="lat")
    for nm, (x0, x1, y0, y1) in {"S of 60S": (0, 360, -80, -60), "60S-30S": (0, 360, -60, -30), "30S-30N": (0, 360, -30, 30),
                                 "N of 30N": (0, 360, 30, 70), **{f"{b} box": BOX[b] for b in WBC}}.items():
        m = (LON >= x0) & (LON < x1) & (LAT >= y0) & (LAT < y1)
        rows.append({"region": nm, "run": k, "dV(σ2>37.0) Jan->Jun (1e15 m3)": np.nansum(dV[m]) / 1e15})
plt.tight_layout()
mkmd.savefig(fig, "Dense water loss 8km", "Change in the thickness of water denser than σ2 = 37.0 between January and June 1900. In 8km it is lost in the boundary-current extensions and the ACC band; 25km shows no such loss.", dpi=90)
pd.DataFrame(rows).pivot(index="region", columns="run").round(2)

Global volume denser than σ2, 1e15 m3


8km                                                 25km  \
           V(σ2>35.5) V(σ2>36.0) V(σ2>36.8) V(σ2>37.0)    total V(σ2>35.5)   
month 1900                                                                   
1             1238.13    1168.01     864.40     528.69  1334.65    1238.24   
2             1238.45    1168.39     863.34     527.24  1334.65    1238.17   
3             1239.20    1169.54     860.98     523.65  1334.64    1238.14   
4             1240.27    1171.49     857.28     518.09  1334.64    1238.10   
5             1241.43    1174.01     852.49     511.20  1334.64    1238.11   
6             1242.87    1176.53     847.34     504.38  1334.64    1238.21   

                                                      
           V(σ2>36.0) V(σ2>36.8) V(σ2>37.0)    total  
month 1900                                            
1             1168.02     864.12     527.83  1334.63  
2             1167.90     864.36     527.88  1334.62  
3             1167.86     864.39     527.88  1334.62  
4             1167.87     864.50     527.87  1334.61  
5             1167.93     864.65     527.93  1334.61  
6             1168.01     864.75     528.13  1334.61

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_05.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.


dV(σ2>37.0) Jan->Jun (1e15 m3)       
run                                       25km    8km
region                                               
30S-30N                                   0.15  -2.05
60S-30S                                  -0.12 -18.34
Agulhas box                              -0.06  -3.51
EAC box                                  -0.00  -1.01
Gulf Stream box                           0.01  -1.67
Kuroshio box                              0.02  -1.53
N of 30N                                 -0.16  -3.41
S of 60S                                  0.37  -0.55

### B5. Numerical mixing diagnostics
`T_advection_scheme_variance_production` is the temperature variance created or destroyed by the tracer advection scheme; negative means numerical mixing. `T_vardec` is the variance change from vertical remapping (ALE). Both are integrated over depth bands and averaged over each box. Units: K² m s⁻¹.

**Caveat when comparing:** the 8km run has `KHTR` = 0 and no GM, so the advection numerics are its only sink for lateral tracer variance. 25km removes that variance explicitly instead (`KHTR` = 50 neutral diffusion, plus GM). A larger numerical term at 8km is therefore partly expected. These diagnostics alone do not show how much of that mixing is diapycnal; B4 shows that diapycnal change does happen.

In [16]:
rows = []
for v in ["T_advection_scheme_variance_production", "T_vardec"]:
    for nm in WBC:
        for k, E in P2.items():
            A, _ = area_lat(E); sx, sy = box_slices(E, *BOX[nm]); a = A[sy, sx]
            for mo in [0, 2, 5]:
                f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.e.1mon.mean.1900.nc")[mo]
                with xr.open_dataset(f, decode_timedelta=False) as ds:
                    e = ds.e.isel(time=i, yh=sy, xh=sx).values
                h = e[:-1] - e[1:]; zc = -0.5 * (e[:-1] + e[1:])
                f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.{v}.1mon.mean.1900.nc")[mo]
                with xr.open_dataset(f, decode_timedelta=False) as ds:
                    q = ds[v].isel(time=i, yh=sy, xh=sx).values.astype(float)
                col = q * h * a[None]
                rows.append({"diagnostic": v, "box": nm, "run": k, "month": mo + 1,
                             **{f"{z0}-{z1} m": np.nansum(np.where((zc >= z0) & (zc < z1), col, 0)) / a.sum() for z0, z1 in BANDS}})
pd.DataFrame(rows).set_index(["diagnostic", "box", "run", "month"]).style.format("{:.1e}")

### B6. Tracer-advection CFL estimate
Monthly-maximum surface speed × `DT_TRACER_ADVECT` / min(dx, dy). From month 3, 8km's fastest current cells sit at about twice the 25km CFL. This is a surface estimate; currents at depth are slower.

In [17]:
DTA = {"8km": 3600.0, "25km": 7200.0}; rows = []
for k, E in P2.items():
    st = static(E); dmin = np.minimum(st.dxt.values, st.dyt.values); wet = st.wet.values > 0
    lon = (st.xh.values + 360) % 360; lat = st.yh.values
    for mo in [0, 2, 5]:
        f, i, _ = month_index(f"{E}/output*/access-om3.mom6.2d.speed.1mon.max.1900.nc")[mo]
        with xr.open_dataset(f, decode_timedelta=False) as ds:
            sp = ds[list(ds.data_vars)[0]].isel(time=i).values
        cfl = sp * DTA[k] / dmin
        for nm in WBC:
            x0, x1, y0, y1 = BOX[nm]; ix = (lon >= x0) & (lon < x1); iy = (lat >= y0) & (lat < y1)
            c = cfl[np.ix_(iy, ix)][wet[np.ix_(iy, ix)]]
            rows.append({"box": nm, "run": k, "month": mo + 1, "CFL p99": np.nanpercentile(c, 99), "CFL max": np.nanmax(c),
                         "fraction CFL > 0.5": np.nanmean(c > 0.5)})
pd.DataFrame(rows).set_index(["box", "run", "month"]).round(3)

,,,CFL p99,CFL max,fraction CFL > 0.5
box,run,month,,,
Gulf Stream,8km,1,0.944,1.232,0.168
Kuroshio,8km,1,0.517,0.923,0.012
Agulhas,8km,1,1.104,1.173,0.520
EAC,8km,1,0.629,0.783,0.050
Gulf Stream,8km,3,0.830,1.288,0.154
Kuroshio,8km,3,0.741,0.987,0.064
Agulhas,8km,3,0.817,1.293,0.144
EAC,8km,3,0.712,1.002,0.054
Gulf Stream,8km,6,0.687,0.849,0.085


# Part C: candidate causes checked

### C1. Wind stress the ocean receives (1909 annual mean, binned to 1°)

In [18]:
tau = {(k, v): bin_lat_lon(annual(E, v)) for k, E in RUNS.items() for v in ["tauuo", "tauvo"]}

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i, v in enumerate(["tauuo", "tauvo"]):
    for k in RUNS:
        tau[(k, v)].mean("lon").plot(ax=ax[i], label=k)
    a, b = tau[("8km", v)], tau[("25km", v)]; m = np.isfinite(a) & np.isfinite(b)
    r = np.corrcoef(a.values[m], b.values[m])[0, 1]
    ax[i].set(title=f"zonal-mean {v} (pattern corr = {r:.3f})", xlim=(-80, 65), ylabel="N m$^{-2}$"); ax[i].legend()
(tau[("8km", "tauuo")] - tau[("25km", "tauuo")]).plot(ax=ax[2], vmin=-0.05, vmax=0.05, cmap="RdBu_r")
ax[2].set(title="tauuo 8km - 25km", ylim=(-80, 65))
plt.tight_layout()
mkmd.savefig(fig, "Wind stress 8km vs 25km", "1909 annual-mean surface stress received by the ocean. On a 1° grid the two runs are nearly identical.", dpi=100)

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_06.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.


### C2. Wind-stress curl at native resolution (1909 annual mean)
Curl on each model's own grid (C-grid differences at corner points). If remapping to the 8km grid added grid-scale noise, the grid-scale part (half the rms of the 2-point x-difference) would be larger at 8km.

In [19]:
def curl(E):
    st = static(E)
    tx = annual(E, "tauuo").values * st.dxCu.values; ty = annual(E, "tauvo").values * st.dyCv.values
    dtydx = np.roll(ty, -1, axis=1) - ty; dtxdy = np.roll(tx, -1, axis=0) - tx
    ny = min(dtydx.shape[0], dtxdy.shape[0]); nx = min(dtydx.shape[1], dtxdy.shape[1])
    c = (dtydx[:ny, :nx] - dtxdy[:ny, :nx]) / st.areacello_bu.values[:ny, :nx]
    return xr.DataArray(c, dims=("y", "x"), coords=dict(y=st.yq.values[:ny], x=(st.xq.values[:nx] + 360) % 360))

C = {k: curl(E) for k, E in RUNS.items()}
regs = {"Agulhas": ((10, 40), (-45, -25)), "Eq. Pacific": ((180, 240), (-10, 10)), "Kuroshio": ((125, 165), (25, 45))}
fig, ax = plt.subplots(3, 2, figsize=(12, 11)); rows = []
for i, (n, ((x0, x1), (y0, y1))) in enumerate(regs.items()):
    for j, k in enumerate(RUNS):
        s = C[k].where((C[k].x > x0) & (C[k].x < x1) & (C[k].y > y0) & (C[k].y < y1), drop=True) * 1e7
        s.plot(ax=ax[i, j], vmin=-3, vmax=3, cmap="RdBu_r", cbar_kwargs=dict(label="10$^{-7}$ N m$^{-3}$"))
        ax[i, j].set_title(f"{k} wind-stress curl, {n}, 1909")
        v = s.values
        rows.append(dict(region=n, run=k, rms_curl=np.nanstd(v), grid_scale_rms=np.nanstd(np.diff(v, axis=1)) / 2))
plt.tight_layout()
pd.DataFrame(rows).set_index(["region", "run"]).round(3)

rms_curl  grid_scale_rms
region      run                           
Agulhas     8km      1.371           0.116
            25km     1.388           0.212
Eq. Pacific 8km      0.296           0.011
            25km     0.270           0.013
Kuroshio    8km      1.033           0.082
            25km     1.011           0.169

### C3. Initial conditions
Zonal-mean T of the two WOA23 initial-condition files, and of the January 1900 monthly mean, 8km vs 25km test5.1.

In [20]:
IC = {"8km": "/g/data/vk83/prerelease/configurations/inputs/access-om3/mom/initial_conditions/global.8km/2026.10.07/woa23_ts_01_mom.nc",
      "25km": "/g/data/vk83/configurations/inputs/access-om3/mom/initial_conditions/global.25km/2026.06.11/woa23_ts_01_mom.nc"}
ICZ = {}
for k, E in {"8km": E8, "25km": E25S}.items():
    A, lat = area_lat(E)
    ic = xr.open_dataset(IC[k], decode_times=False).ptemp
    ICZ[(k, "initial condition")] = zm3(ic.isel(time=0) if "time" in ic.dims else ic, A, lat)
    f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.thetao+z.1mon.mean.1900.nc")[0]
    with xr.open_dataset(f, decode_timedelta=False) as ds:
        ICZ[(k, "Jan 1900 mean")] = zm3(ds.thetao.isel(time=i), A, lat)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5)); rows = []
for i, w in enumerate(["initial condition", "Jan 1900 mean"]):
    a, b = ICZ[("8km", w)], ICZ[("25km", w)]
    dd = a - b.interp(z=a.z)
    dd.plot.contourf(ax=ax[i], levels=np.arange(-1, 1.01, 0.1), cmap="RdBu_r", yincrease=False, extend="both")
    ax[i].set_title(f"zonal-mean T 8km - 25km test5.1: {w}")
    for z0, z1 in [(0, 100), (100, 500), (500, 1500), (1500, 6000)]:
        s = dd.sel(z=slice(z0, z1))
        rows.append(dict(field=w, depth=f"{z0}-{z1} m", mean_diff=float(s.mean()), max_abs_diff=float(abs(s).max())))
plt.tight_layout()
pd.DataFrame(rows).set_index(["field", "depth"]).round(3)

mean_diff  max_abs_diff
field             depth                               
initial condition 0-100 m         -0.013         0.275
                  100-500 m       -0.015         0.251
                  500-1500 m      -0.009         0.173
                  1500-6000 m     -0.005         0.082
Jan 1900 mean     0-100 m         -0.035         0.301
                  100-500 m       -0.015         0.196
                  500-1500 m       0.006         0.107
                  1500-6000 m      0.002         0.480

### C4. Runtime MOM6 parameters
Read from each run's `MOM_parameter_doc.all`, which lists every parameter value MOM6 actually used, defaults included. Paths, layouts and processor counts are left out. The second table lists every parameter that **changes between output segments**, in any of the three runs.

In [21]:
SKIP = re.compile(r"FILE|_DIR|INPUTDIR|LAYOUT|MASKTABLE|NIPROC|NJPROC|NIGLOBAL|NJGLOBAL|TOPO|GRID_FILE|CHL_FILE")

def params(path):
    out = {}
    for line in open(path):
        if line.lstrip().startswith("!") or "=" not in line:
            continue
        k, v = line.split("!")[0].split("=", 1)
        out[k.strip()] = " ".join(v.split())
    return out

P = {k: params(f"{E}/output000/MOM_parameter_doc.all") for k, E in ALL.items()}
keys = sorted(k for k in set().union(*P.values()) if not SKIP.search(k) and not k.startswith("MEKE_"))
tab = pd.DataFrame({r: [P[r].get(k, "-") for k in keys] for r in P}, index=pd.Index(keys, name="parameter"))
diff = tab[tab.nunique(axis=1) > 1]
print("MEKE_* parameters (only active in the 25km runs) are omitted.")
diff

MEKE_* parameters (only active in the 25km runs) are omitted.


,8km test2,25km test5.1 (same exe),25km test3
parameter,,,
ADVECTION_SCHEME_VARIANCE_UNDERFLOW,1.0E-23,1.0E-23,-
BODNER_HMIN_BUG,True,True,-
DT,240.0,900.0,900.0
DT_THERM,3600.0,7200.0,7200.0
DT_TRACER_ADVECT,3600.0,7200.0,7200.0
EKE_SOURCE,-,"""prog""","""prog"""
KD,1.5E-05,8.0E-06,1.5E-05
KD_KAPPA_SHEAR_0,1.5E-05,8.0E-06,1.5E-05
KD_MIN_TR,1.5E-06,8.0E-07,1.5E-06


In [22]:
chg = []
for r, E in ALL.items():
    segs = sorted(glob.glob(f"{E}/output*/MOM_parameter_doc.all")); prev = None
    for s in segs:
        p = params(s)
        if prev is not None:
            for k in sorted(set(p) | set(prev[1])):
                a, b = prev[1].get(k), p.get(k)
                if a != b and a is not None and b is not None:   # skip init-only parameters that vanish after output000
                    chg.append(dict(run=r, segment=os.path.basename(os.path.dirname(s)), parameter=k, old=a, new=b))
        prev = (s, p)
chg = pd.DataFrame(chg)
# Model year each 8km segment starts in, from its first monthly hfds file
first_year = {os.path.basename(os.path.dirname(f)): int(f[-7:-3]) for f in
              sorted(glob.glob(f"{E8}/output*/access-om3.mom6.2d.hfds.1mon.mean.*.nc"))[::-1]}
chg["starts in"] = [first_year.get(s, "") if r == "8km test2" else "" for r, s in zip(chg.run, chg.segment)]
chg

,run,segment,parameter,old,new,starts in
0,8km test2,output001,DT,240.0,400.0,1900
1,8km test2,output003,RESTART_CHECKSUMS_REQUIRED,True,False,1900
2,8km test2,output004,RESTART_CHECKSUMS_REQUIRED,False,True,1900
3,8km test2,output011,DT,400.0,600.0,1902
4,8km test2,output017,DT,600.0,720.0,1904
5,8km test2,output020,NONBLOCKING_P_SURF_BUG,True,False,1905


In [23]:
# Same executable and inputs in every 8km segment? (payu manifests)
import hashlib
for kind in ["exe", "input"]:
    h = {hashlib.md5(open(f, "rb").read()).hexdigest() for f in glob.glob(f"{E8}/output*/manifests/{kind}.yaml")}
    print(f"8km {kind} manifest: {len(h)} distinct version(s) across {len(glob.glob(f'{E8}/output*/manifests'))} segments")
# Is the non-blocking p_surf bug code path active? (it needs NONBLOCKING_UPDATES and USE_PSURF_IN_EOS)
for r, E in ALL.items():
    lay = params(f"{E}/output000/MOM_parameter_doc.layout")
    print(f"{r:26s} NONBLOCKING_UPDATES={lay.get('NONBLOCKING_UPDATES', 'False (default)'):6s}"
          f" USE_PSURF_IN_EOS={P[r].get('USE_PSURF_IN_EOS', '-')}")
# Vertical grid: the configs point at different ocean_vgrid.nc paths; compare the interfaces themselves
VG = {"8km": "/g/data/vk83/prerelease/configurations/inputs/access-om3/mom/grids/vertical/share/2025.03.12/ocean_vgrid.nc",
      "25km test5.1": "/g/data/tm70/aek156/make_om3_topo/ocean_vgrid.nc"}
za, zb = (xr.open_dataset(v).zeta.values for v in VG.values())
print(f"ocean_vgrid.nc: {za.size} vs {zb.size} interfaces, max |difference| = {np.abs(za - zb).max():.3g} m")

8km exe manifest: 1 distinct version(s) across 32 segments


8km input manifest: 1 distinct version(s) across 32 segments
8km test2                  NONBLOCKING_UPDATES=True   USE_PSURF_IN_EOS=True
25km test5.1 (same exe)    NONBLOCKING_UPDATES=True   USE_PSURF_IN_EOS=True
25km test3                 NONBLOCKING_UPDATES=False  USE_PSURF_IN_EOS=True
ocean_vgrid.nc: 76 vs 76 interfaces, max |difference| = 0 m


### C5. Physical diapycnal diffusivity: Pacific median profiles (1909 annual mean)

In [24]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for i, (la, lon0, lon1) in enumerate([(0.0, 160, 260), (-30.0, 190, 260)]):
    for k, E in RUNS.items():
        kd = annual(E, "Kd_interface", sel={"yh": la}, dim="3d"); e = annual(E, "e", sel={"yh": la}, dim="3d")
        lon = (e.xh + 360) % 360; m = (lon > lon0) & (lon < lon1)
        ax[i].semilogx(kd.where(m).median("xh"), -e.where(m).median("xh"), label=k)
    ax[i].set(ylim=(1500, 0), xlim=(1e-6, 1e-1), title=f"median Kd_interface, {la}°N, {lon0}-{lon1}E", xlabel="m$^2$ s$^{-1}$", ylabel="depth (m)")
    ax[i].legend()
plt.tight_layout()

# Part D: EAC spin-up movie (1900-1907)
Side by side, 8km test2 vs 25km test5.1 (same executable), one frame every 4 days:
- daily surface speed over eastern Australia
- monthly T and S across the Tasman Sea at 30°S
- global-mean T, global KE and box-mean SST, with a moving cursor

Adapted from Andy Hogg's [13-ACCESS-OM3-SSS-Movie](https://github.com/AndyHoggANU/ACCESS-Visualisation-Recipes/blob/AH-testing/13-ACCESS-OM3-SSS-Movie.ipynb). Rendering takes about 20-30 min, so it only runs when `MAKE_MOVIE = True`; otherwise the existing mp4 is shown.

In [25]:
MAKE_MOVIE = False
MOVIE = "eac_spinup_8km_vs_25km.mp4"
MYEARS = list(range(1900, 1908)); STRIDE = 4
MBOX = dict(lon=(145, 172), lat=(-45, -15))
SEC_LAT, SEC_LON, SEC_ZMAX = -30.0, (152.5, 172), 1000
MRUNS = {"8km test2": E8, "25km test5.1": E25S}

def box_index(ds, lon, lat):
    x = (ds.xh.values + 360) % 360
    ix = np.where((x >= lon[0]) & (x <= lon[1]))[0]; iy = np.where((ds.yh.values >= lat[0]) & (ds.yh.values <= lat[1]))[0]
    return slice(ix.min(), ix.max() + 1), slice(iy.min(), iy.max() + 1)

def load_daily(exp, var):
    das = []
    for yr in MYEARS:
        for f in sorted(glob.glob(f"{exp}/output*/access-om3.mom6.2d.{var}.1day.mean.{yr}.nc")):
            ds = xr.open_dataset(f, decode_timedelta=False); sx, sy = box_index(ds, **MBOX)
            das.append(ds[var].isel(xh=sx, yh=sy).load())
    da = xr.concat(das, "time").drop_duplicates("time").sortby("time")
    return da.assign_coords(xh=(da.xh + 360) % 360)

def load_section(exp, var):
    das = []
    for yr in MYEARS:
        for f in sorted(glob.glob(f"{exp}/output*/access-om3.mom6.3d.{var}+z.1mon.mean.{yr}.nc")):
            ds = xr.open_dataset(f, decode_timedelta=False); sx, _ = box_index(ds, SEC_LON, (-90, 90))
            das.append(ds[var].sel(yh=SEC_LAT, method="nearest").isel(xh=sx).sel(z_l=slice(0, SEC_ZMAX)).load())
    da = xr.concat(das, "time").drop_duplicates("time").sortby("time")
    return da.assign_coords(xh=(da.xh + 360) % 360)

def frac_year(t):
    return t.dt.year.values + (t.dt.dayofyear.values - 0.5) / 365

In [26]:
if MAKE_MOVIE:
    import shutil, matplotlib.animation as animation, cmocean as cm
    speed = {k: load_daily(E, "speed") for k, E in MRUNS.items()}
    tos = {k: load_daily(E, "tos") for k, E in MRUNS.items()}
    secT = {k: load_section(E, "thetao") for k, E in MRUNS.items()}
    secS = {k: load_section(E, "so") for k, E in MRUNS.items()}
    common = np.intersect1d(speed["8km test2"].time.values, speed["25km test5.1"].time.values)[::STRIDE]
    box_sst = {k: v.mean(["xh", "yh"]) for k, v in tos.items()}
    colors = {"8km test2": "C3", "25km test5.1": "C0"}
    fig = plt.figure(figsize=(14, 17), facecolor="w")
    gs = fig.add_gridspec(4, 6, height_ratios=[2.2, 1, 1, 1], hspace=0.35, wspace=0.6)
    t0 = common[0]; im_sp, im_T, im_S, cursors = {}, {}, {}, []
    for c, k in enumerate(MRUNS):
        ax = fig.add_subplot(gs[0, 3 * c:3 * c + 3]); ax.set_facecolor("0.6")
        sp = speed[k].sel(time=t0)
        im_sp[k] = ax.pcolormesh(sp.xh, sp.yh, sp, vmin=0, vmax=1.2, cmap=cm.cm.thermal, shading="auto")
        ax.plot(SEC_LON, [SEC_LAT] * 2, "w--", lw=1)
        ax.set(title=f"{k}: surface speed", xlabel="lon", ylabel="lat"); ax.set_aspect(1 / np.cos(np.deg2rad(30)))
        fig.colorbar(im_sp[k], ax=ax, label="m s$^{-1}$", shrink=0.8)
        for r, (sec, dd, vmin, vmax, cmap, lab) in enumerate([(secT, im_T, 4, 26, cm.cm.thermal, "°C"),
                                                              (secS, im_S, 34.4, 35.8, cm.cm.haline, "psu")]):
            ax = fig.add_subplot(gs[1 + r, 3 * c:3 * c + 3]); ax.set_facecolor("0.6")
            s = sec[k].sel(time=t0, method="nearest")
            dd[k] = ax.pcolormesh(s.xh, s.z_l, s, vmin=vmin, vmax=vmax, cmap=cmap, shading="auto")
            ax.invert_yaxis(); fig.colorbar(dd[k], ax=ax, label=lab)
            ax.set(title=f"{k}: monthly {'T' if r == 0 else 'S'} at {SEC_LAT:.0f}°N", ylabel="depth (m)")
    for i, (title, ylab) in enumerate([("global-mean T (ocean.stats)", "°C"),
                                       ("global KE per unit mass (ocean.stats)", "m$^2$ s$^{-2}$"),
                                       ("box-mean SST (map region)", "°C")]):
        ax = fig.add_subplot(gs[3, 2 * i:2 * i + 2])
        for k, E in MRUNS.items():
            if i < 2:
                day, T, En = stats(E); yr = 1900 + day / 365; m = yr <= MYEARS[-1] + 1
                ax.plot(yr[m], (T if i == 0 else En)[m], color=colors[k], label=k)
            else:
                ax.plot(frac_year(box_sst[k].time), box_sst[k], color=colors[k], label=k)
        ax.set(title=title, ylabel=ylab, xlim=(MYEARS[0], MYEARS[-1] + 1)); ax.legend(fontsize=8)
        cursors.append(ax.axvline(MYEARS[0], color="k", lw=1))
    clock = fig.suptitle("", fontsize=18, y=0.93)

    def update(t):
        for k in MRUNS:
            im_sp[k].set_array(speed[k].sel(time=t).values.ravel())
            im_T[k].set_array(secT[k].sel(time=t, method="nearest").values.ravel())
            im_S[k].set_array(secS[k].sel(time=t, method="nearest").values.ravel())
        x = frac_year(xr.DataArray([t], dims="time"))[0]
        for cl in cursors:
            cl.set_xdata([x, x])
        clock.set_text(f"{t.year:04d}-{t.month:02d}-{t.day:02d}")
        return []

    plt.rcParams["animation.ffmpeg_path"] = shutil.which("ffmpeg")
    animation.FuncAnimation(fig, update, frames=common, blit=False).save(
        MOVIE, writer=animation.FFMpegWriter(fps=15, bitrate=4000), dpi=80)
    plt.close(fig)

from IPython.display import Video, display
if os.path.exists(MOVIE):
    display(Video(MOVIE, embed=False, width=800))
else:
    print(f"{MOVIE} not found next to this notebook; set MAKE_MOVIE = True to render it.")

# Part E: one-change test, `DT_THERM` 3600 → 2400 s
Per km of grid spacing, the 8km run's time steps are longer than the 25km ones: `DT_THERM` 3600 s / 8 km = 450 s/km vs 7200 s / 25 km = 288 s/km. `DT_THERM` was 3600 s in all 32 segments of test2, so it has never been varied. The test repeats test2 from the same WOA23 initial state with **only** `DT_THERM` changed (the tracer advection step follows it by default). Its first segment copies test2's `output000` exactly (2 months, `DT` = 240 s). It may be extended to 6 months, following test2's `output001`-`output002` (`DT` = 400 s).

Config: `access-om3-configs` branch `MC_8km_jra_ryf-test2` (8165f53) + test2's as-run `output000` `config.yaml`, `nuopc.runconfig` and exe manifest, local branch `8km-dtherm2400-test`.

In [27]:
TEST = sorted(glob.glob("/g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/lab/archive/*dtherm2400*"))
if not TEST or not glob.glob(f"{TEST[0]}/output*/ocean.stats"):
    print("DT_THERM test output not available yet.")
else:
    ET = TEST[0]
    dT_, TT, EnT = stats(ET); d8, T8, En8 = ST["8km test2"]; d25, T25, En25 = ST["25km test5.1 (same exe)"]
    dmax = dT_.max()
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for lab, (d, T, En) in {"8km test2 (DT_THERM 3600)": (d8, T8, En8), "8km DT_THERM 2400": (dT_, TT, EnT),
                            "25km test5.1": (d25, T25, En25)}.items():
        m = d <= dmax
        ax[0].plot(d[m], T[m] - T[0], label=lab); ax[1].plot(d[m], En[m], label=lab)
    ax[0].set(title="global-mean T change since 1 Jan 1900", xlabel="day", ylabel="°C")
    ax[1].set(title="global KE per unit mass", xlabel="day", ylabel="m$^2$ s$^{-2}$")
    for a in ax: a.legend()
    plt.tight_layout()
    mkmd.savefig(fig, "DT_THERM test", "Global-mean T and KE in the first months: 8km test2, the same run with DT_THERM = 2400 s, and 25km test5.1.", dpi=100)
    chk = [d for d in [30, 60, 90, 120, 150, 180] if d <= dmax]
    tab = pd.DataFrame({"test2 dT": np.interp(chk, d8, T8 - T8[0]), "DT_THERM 2400 dT": np.interp(chk, dT_, TT - TT[0]),
                        "25km test5.1 dT": np.interp(chk, d25, T25 - T25[0])}, index=pd.Index(chk, name="day"))
    # Monthly global and 10S-10N hfds
    A, lat = area_lat(E8); eqm = (lat > -10) & (lat < 10)
    rows = []
    for lab, E in {"test2": E8, "DT_THERM 2400": ET}.items():
        for j, (f, i, t) in enumerate(month_index(f"{E}/output*/access-om3.mom6.2d.hfds.1mon.mean.1900.nc")):
            if j * 30 >= dmax: break
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                h = ds.hfds.isel(time=i).values
            rows.append(dict(run=lab, month=j + 1, global_PW=np.nansum(h * A) / 1e15,
                             eq_10S10N_Wm2=np.nansum((h * A)[eqm]) / np.nansum(A[eqm])))
    display(tab.style.format("{:+.5f}"))
    display(pd.DataFrame(rows).pivot(index="month", columns="run").round(2))

Saved /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis_07.png
Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md
Lines appended to /g/data/tm70/cyb561/repos/claude/what-is-wrong-with-8k/scratch/paper1/notebooks/mkfigs_output_test/mkmd/8km_ryf_drift_diagnosis.md successfully.


,test2 dT,DT_THERM 2400 dT,25km test5.1 dT
day,,,
30,+0.00330,+0.00330,+0.00310
60,+0.00565,+0.00545,+0.00480
90,+0.00790,+0.00730,+0.00620
120,+0.00850,+0.00740,+0.00590


global_PW       eq_10S10N_Wm2       
run   DT_THERM 2400 test2 DT_THERM 2400  test2
month                                         
1              6.93  6.97         36.70  36.59
2              4.51  4.89         42.91  42.74
3              3.93  4.71         54.55  54.49
4              0.15  1.38         45.68  46.15

### E2. The same test, measured with Part B's diagnostics
For each month the test has: global volume denser than σ₂ = 37.0 (B4), advection-scheme numerical mixing below 700 m in the current-extension boxes (B5), and the box-mean change in 300 m temperature since month 1 (B1). If the shorter `DT_THERM` removes the spurious mixing, the test should sit with 25km, not with test2.

In [28]:
if TEST and glob.glob(f"{TEST[0]}/output*/access-om3.mom6.3d.thkcello+rho2.1mon.mean.1900.nc"):
    ET = TEST[0]
    TRI = {"8km test2": E8, "8km DT_THERM 2400": ET, "25km test5.1": E25S}
    nmon = len(month_index(f"{ET}/output*/access-om3.mom6.3d.thkcello+rho2.1mon.mean.1900.nc"))
    rows = []
    for k, E in TRI.items():
        A = static(E).areacello.fillna(0).values; Aw, _ = area_lat(E)
        ridx = month_index(f"{E}/output*/access-om3.mom6.3d.thkcello+rho2.1mon.mean.1900.nc")
        tidx = month_index(f"{E}/output*/access-om3.mom6.3d.thetao+z.1mon.mean.1900.nc")
        for mo in range(nmon):
            f, i, _ = ridx[mo]
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                h = ds.thkcello.isel(time=i); rd = h.dims[0]; rho = ds[rd].values
                v37 = sum(np.nansum(np.nan_to_num(h.isel({rd: kk}).values) * A) for kk in np.where(rho > 1037.0)[0]) / 1e15
            row = {"run": k, "month": mo + 1, "V(σ2>37.0) 1e15 m3": v37}
            for nm in ["Gulf Stream", "Agulhas"]:
                sx, sy = box_slices(E, *BOX[nm]); a = Aw[sy, sx]
                f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.e.1mon.mean.1900.nc")[mo]
                with xr.open_dataset(f, decode_timedelta=False) as ds:
                    e = ds.e.isel(time=i, yh=sy, xh=sx).values
                hh = e[:-1] - e[1:]; zc = -0.5 * (e[:-1] + e[1:])
                f, i, _ = month_index(f"{E}/output*/access-om3.mom6.3d.T_advection_scheme_variance_production.1mon.mean.1900.nc")[mo]
                with xr.open_dataset(f, decode_timedelta=False) as ds:
                    q = ds.T_advection_scheme_variance_production.isel(time=i, yh=sy, xh=sx).values.astype(float)
                row[f"{nm} adv. num. mixing >700 m (K2 m/s)"] = np.nansum(np.where(zc >= 700, q * hh * a[None], 0)) / a.sum()
                f, i, _ = tidx[mo]
                with xr.open_dataset(f, decode_timedelta=False) as ds:
                    t = ds.thetao.isel(time=i, yh=sy, xh=sx).sel(z_l=300.0, method="nearest").values
                row[f"{nm} T300 box mean (°C)"] = np.nansum(np.nan_to_num(t) * a) / a[np.isfinite(t)].sum()
            rows.append(row)
    e2 = pd.DataFrame(rows).set_index(["month", "run"]).sort_index()
    display(e2.style.format("{:.3g}"))
else:
    print("DT_THERM test output not available yet.")

### E3. Where the remaining extra heat uptake is
Surface heat flux (`hfds`) integrated over latitude bands, in PW, with the difference from 25km test5.1 in brackets.

In [29]:
if TEST and glob.glob(f"{TEST[0]}/output*/access-om3.mom6.2d.hfds.1mon.mean.1900.nc"):
    LB = {"global": (-90, 90), "S of 45S": (-90, -45), "45S-20S": (-45, -20), "20S-20N": (-20, 20), "20N-45N": (20, 45), "N of 45N": (45, 90)}
    rows = []
    for k, E in {"8km test2": E8, "8km DT_THERM 2400": TEST[0], "25km test5.1": E25S}.items():
        A, lat = area_lat(E8 if "8km" in k else E)
        for mo, (f, i, _) in enumerate(month_index(f"{E}/output*/access-om3.mom6.2d.hfds.1mon.mean.1900.nc")[:6]):
            with xr.open_dataset(f, decode_timedelta=False) as ds:
                h = np.nan_to_num(ds.hfds.isel(time=i).values) * A
            for b, (y0, y1) in LB.items():
                rows.append({"run": k, "month": mo + 1, "band": b, "PW": h[(lat >= y0) & (lat < y1)].sum() / 1e15})
    hb = pd.DataFrame(rows)
    nm = hb[hb.run == "8km DT_THERM 2400"].month.max()
    hb = hb[hb.month <= nm].pivot_table(index=["band", "month"], columns="run", values="PW")
    for k in ["8km test2", "8km DT_THERM 2400"]:
        hb[f"{k} - 25km"] = hb[k] - hb["25km test5.1"]
    display(hb.round(2))
else:
    print("DT_THERM test output not available yet.")

run             25km test5.1  8km DT_THERM 2400  8km test2  8km test2 - 25km  \
band     month                                                                 
20N-45N  1             -5.45              -5.37      -5.34              0.12   
         2             -4.07              -3.93      -3.75              0.32   
         3             -0.48              -0.29       0.03              0.51   
         4              3.27               3.42       3.81              0.54   
20S-20N  1              2.79               2.89       2.87              0.08   
         2              3.58               3.79       3.76              0.18   
         3              5.27               5.50       5.47              0.20   
         4              4.15               4.36       4.38              0.23   
45S-20S  1              6.20               6.35       6.36              0.16   
         2              3.96               4.19       4.36              0.39   
         3             -0.21               0.04       0.39              0.60   
         4             -4.85              -4.63      -4.08              0.77   
N of 45N 1             -3.86              -3.83      -3.83              0.03   
         2             -2.91              -2.86      -2.85              0.05   
         3             -1.24              -1.19      -1.19              0.06   
         4              0.37               0.40       0.40              0.03   
S of 45S 1              6.82               6.90       6.91              0.08   
         2              3.13               3.32       3.37              0.24   
         3             -0.36              -0.12       0.01              0.37   
         4             -3.66              -3.39      -3.13              0.53   
global   1              6.50               6.93       6.97              0.47   
         2              3.70               4.51       4.89              1.19   
         3              2.98               3.93       4.71              1.73   
         4             -0.72               0.15       1.38              2.10   

run             8km DT_THERM 2400 - 25km  
band     month                            
20N-45N  1                          0.08  
         2                          0.13  
         3                          0.18  
         4                          0.14  
20S-20N  1                          0.09  
         2                          0.21  
         3                          0.23  
         4                          0.21  
45S-20S  1                          0.15  
         2                          0.22  
         3                          0.25  
         4                          0.22  
N of 45N 1                          0.03  
         2                          0.05  
         3                          0.05  
         4                          0.03  
S of 45S 1                          0.08  
         2                          0.20  
         3                          0.25  
         4                          0.27  
global   1                          0.43  
         2                          0.81  
         3                          0.96  
         4                          0.87